01-lib.ipynb
02-corpus.ipynb
03-vocab.ipynb
04-dtm-tfidf.ipynb
05-pca.ipynb
06-lda.ipynb
07-sentiment.ipynb
08-word2vec.ipynb
09-riffs.ipynb

## Construct CORPUS table

First I am going to look at the files that make up each of the StandardEbooks.


For Giant's Bread which has book and chapter separations, I am going to do the following: as book in the giants bread is a product of publishing (i believe) rather than narrative, I will just continue the numbering making 2-1 be the next number after 1-x ends (REWRITE)

In [101]:
# import libraries
import pandas as pd
import glob
import os
import re
import nltk
import xml.etree.ElementTree as ET
from pathlib import Path
from bs4 import BeautifulSoup

In [102]:
nltk_resources = [
    'punkt_tab',
    'averaged_perceptron_tagger_eng',
    'stopwords',
    'tagsets'
]

for resource in nltk_resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Could not download {resource}: {e}")

In [103]:
# get list of all files in the raw_data folder
root_dir = 'raw_data'
file_dict = {}

# get paths to xhtml files
paths = sorted(Path(root_dir).glob('*/src/epub/text/*.xhtml')) # glob finds all the pathnames matching a specified pattern

for path in paths:
    book_id = path.parts[1].replace('agatha-christie_', '') # .parts breaks a path into a tuple of components
    # for each book_id, get the file path to all xhtml files
    if book_id not in file_dict:
        file_dict[book_id] = []
    file_dict[book_id].append(path)

file_dict

{'giants-bread': [PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-1.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-2.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-3.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-4.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-5.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-1.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-2.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-3.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-4.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-5.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-6.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/tex

### Write helper functions

In [104]:
BOILERPLATE = {'colophon', 'imprint', 'titlepage', 'uncopyright', 
               'halftitlepage', 'loi', 'dedication'}

def is_content(path):
    stem = path.stem
    if stem in BOILERPLATE:
        return False
    if stem.startswith('book-'):
        return False
    return True

In [105]:
GIANTS_BREAD_CHAPTER_COUNTS = {1: 8, 2: 7, 3: 4, 4: 4, 5: 5}

# make helper function
def get_chap_num(book_id, stem):
    if stem == 'prologue':
        chap_num = 0
    elif book_id == 'giants-bread' and stem.startswith('chapter-'):
        # get the numbers
        N = int(stem.split('-')[1])
        M = int(stem.split('-')[2])
        # cumulative offset for giants bread chapters
        chap_num = sum(count for book, count in GIANTS_BREAD_CHAPTER_COUNTS.items() if book < N) + M
    elif stem.startswith('chapter-'):
        # get the number
        N = int(stem.split('-')[1])
        chap_num = N
    else:
        raise ValueError(f"Cannot determine chap_num for book_id='{book_id}', stem='{stem}'")
    
    return chap_num

In [106]:
# test
print(get_chap_num('the-big-four', 'chapter-5'))        # expect 5
print(get_chap_num('giants-bread', 'chapter-2-3'))      # expect 11
print(get_chap_num('the-secret-adversary', 'prologue')) # expect 0

5
11
0


In [107]:
def parse_file(book_id, chap_num, xhtml_path):
    """
    Returns a TOKENS dataframe for one file.
    Uses some of the logic from the tokenize_source() function from HW 4.
    """
    # open file and parse with BeautifulSoup
    with open(xhtml_path, "r") as f:
        soup = BeautifulSoup(f, 'xml')
    # find all <p> tags to get list of paragraph text and filter out chap title
    text_paras = [p.get_text() for p in soup.find_all('p')
                  if 'title' not in p.get('epub:type', '')] # this is a list

    # create paragraph dataframe (must wrap the list in a series to use to_frame)
    PARAS = pd.Series(text_paras).to_frame('para_str')

    # parse into sentences with NLTK's sent_tokenize() method
    SENTS = PARAS.para_str.apply(lambda x: pd.Series(nltk.sent_tokenize(x), dtype = 'string'))\
        .stack().to_frame('sent_str')
    # cleanup (don't need PARAS df object anymore)
    del(PARAS)
    # drop NaN values from sents (empty paragraphs that became NaN after stack operation)
    SENTS = SENTS.dropna(subset=['sent_str'])

    # parse into tokens with NLTK's word_tokenize() method
    TOKENS = SENTS.sent_str.apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
        .stack().to_frame('pos_tuple')
    # drop NaN values (empty tuples that became NaN after stack operation)
    TOKENS = TOKENS.dropna(subset=['pos_tuple'])
    TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
    TOKENS['pos_group'] = TOKENS.pos.str[:2]
    TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
    # cleanup: don't need pos_tuple anymore
    TOKENS = TOKENS.drop('pos_tuple', axis = 1)
    TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"[\W_]+", "", regex = True)
    TOKENS = TOKENS[TOKENS.term_str != ''].copy() # copy() ensures working on a dataframe rather than a view of one; good to include after bool filtering
    # cleanup (don't need SENTS df object anymore)
    del(SENTS)

    # add the chap_num and book_id (which are constant per file)
    TOKENS['chap_num'] = chap_num
    TOKENS['book_id'] = book_id

    # name, set, and order indices
    TOKENS.index.names = ['para_num', 'sent_num', 'token_num']
    TOKENS = TOKENS.set_index(['book_id', 'chap_num'], append=True)
    TOKENS = TOKENS.reorder_levels(['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num'])

    return TOKENS

In [ ]:
# test
parse_file('the-big-four', 1, 'raw_data/agatha-christie_the-big-four/src/epub/text/chapter-1.xhtml')

FrozenList(['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num'])

### Main parsing loop

In [110]:
# for each book_id in file_dict
    # for each content file
        # for each paragraph in that file
    